# Model 3: RAG (Retrieval-Augmented Generation)

**Goal:** For each question, first *find* the relevant Austrian tax law text, then give it to GPT as context.

**3 steps:**
1. Load Austrian tax law texts from local PDF files (KStG, EStG, UStG)
2. Embed the law paragraphs as vectors so we can search them
3. For each question: find the most relevant paragraphs → ask GPT with that context

In [1]:
# Install required packages (run once)
!pip install openai pandas numpy pdfplumber

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 20.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 34.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pdfplumber]3 [pdfplumber]


In [ ]:
import pandas as pd
import numpy as np
import re
import pdfplumber
from openai import OpenAI

API_KEY = "your-api-key-here"
client = OpenAI(api_key=API_KEY)

## Step 1: Load Austrian Tax Law Texts from Local PDFs

We read the 3 law texts directly from the PDF files saved in `Context/Gesetze/`.

In [ ]:
def read_pdf(path):
    """Extract all text from a PDF file."""
    text = ""
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    print(f"Read {path.split('/')[-1]}: {len(text)} characters")
    return text

kstg_text = read_pdf("../Context/Gesetze/KStG 1988, Fassung vom 03.04.2026.pdf")
estg_text = read_pdf("../Context/Gesetze/EStG.pdf")
ustg_text = read_pdf("../Context/Gesetze/UStG.pdf")

In [4]:
def split_into_chunks(text, min_length=100):
    """Split law text into paragraphs. Each paragraph starts with a § symbol."""
    chunks = re.split(r'(?=§\s*\d+)', text)
    chunks = [c.strip() for c in chunks if len(c.strip()) > min_length]
    return chunks

kstg_chunks = split_into_chunks(kstg_text)
estg_chunks = split_into_chunks(estg_text)
ustg_chunks = split_into_chunks(ustg_text)
all_chunks  = kstg_chunks + estg_chunks + ustg_chunks

print(f"KStG chunks: {len(kstg_chunks)}")
print(f"EStG chunks: {len(estg_chunks)}")
print(f"UStG chunks: {len(ustg_chunks)}")
print(f"Total chunks: {len(all_chunks)}")
print("\nExample chunk:")
print(all_chunks[0][:300])

KStG chunks: 468
EStG chunks: 2060
UStG chunks: 425
Total chunks: 2953

Example chunk:
Bundesrecht konsolidiert
Gesamte Rechtsvorschrift für Körperschaftsteuergesetz 1988, Fassung vom 03.04.2026
Langtitel
Bundesgesetz vom 7. Juli 1988 über die Besteuerung des Einkommens von Körperschaften
(Körperschaftsteuergesetz 1988 – KStG 1988)
StF: BGBl. Nr. 401/1988 (NR: GP XVII RV 622 AB 674 S.


## Step 2: Embed the Law Paragraphs

We convert each law paragraph into a vector (a list of numbers) using OpenAI's embedding model.
Later, we do the same for each question and find which paragraphs are closest = most relevant.

In [5]:
def get_embedding(text):
    """Convert text to a vector using OpenAI's embedding model."""
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text[:2000]
    )
    return response.data[0].embedding

print(f"Embedding {len(all_chunks)} law paragraphs... (this takes a few minutes)")

chunk_embeddings = []
for i, chunk in enumerate(all_chunks):
    emb = get_embedding(chunk)
    chunk_embeddings.append(emb)
    if (i + 1) % 20 == 0:
        print(f"  {i+1}/{len(all_chunks)} done")

chunk_embeddings = np.array(chunk_embeddings)
print(f"\nDone! Shape: {chunk_embeddings.shape}")

Embedding 2953 law paragraphs... (this takes a few minutes)
  20/2953 done
  40/2953 done
  60/2953 done
  80/2953 done
  100/2953 done
  120/2953 done
  140/2953 done
  160/2953 done
  180/2953 done
  200/2953 done
  220/2953 done
  240/2953 done
  260/2953 done
  280/2953 done
  300/2953 done
  320/2953 done
  340/2953 done
  360/2953 done
  380/2953 done
  400/2953 done
  420/2953 done
  440/2953 done
  460/2953 done
  480/2953 done
  500/2953 done
  520/2953 done
  540/2953 done
  560/2953 done
  580/2953 done
  600/2953 done
  620/2953 done
  640/2953 done
  660/2953 done
  680/2953 done
  700/2953 done
  720/2953 done
  740/2953 done
  760/2953 done
  780/2953 done
  800/2953 done
  820/2953 done
  840/2953 done
  860/2953 done
  880/2953 done
  900/2953 done
  920/2953 done
  940/2953 done
  960/2953 done
  980/2953 done
  1000/2953 done
  1020/2953 done
  1040/2953 done
  1060/2953 done
  1080/2953 done
  1100/2953 done
  1120/2953 done
  1140/2953 done
  1160/2953 done
  1180/

## Step 3: Run RAG Inference

For each question:
1. Embed the question as a vector
2. Find the 3 law paragraphs most similar to the question (cosine similarity)
3. Give those paragraphs to GPT as context, then ask GPT the question

In [6]:
def find_relevant_chunks(question, k=3):
    """Find the k most relevant law paragraphs for a question."""
    q_emb = np.array(get_embedding(question))
    scores = chunk_embeddings @ q_emb / (
        np.linalg.norm(chunk_embeddings, axis=1) * np.linalg.norm(q_emb) + 1e-9
    )
    top_k_idx = np.argsort(scores)[-k:][::-1]
    return [all_chunks[i] for i in top_k_idx]

# Test the retrieval
test_q = "Was ist die Bemessungsgrundlage für die Körperschaftsteuer?"
test_result = find_relevant_chunks(test_q)
print(f"Test question: {test_q}")
print(f"\nMost relevant paragraph (first 300 chars):")
print(test_result[0][:300])

Test question: Was ist die Bemessungsgrundlage für die Körperschaftsteuer?

Most relevant paragraph (first 300 chars):
§ 7. (1) Der Körperschaftsteuer ist das Einkommen zugrunde zu legen, das der unbeschränkt
Steuerpflichtige innerhalb eines Kalenderjahres bezogen hat.
(2) Einkommen ist der Gesamtbetrag der Einkünfte aus den im


In [ ]:
# Load the dataset
df = pd.read_csv("../dataset_clean.csv")
print(f"Loaded {len(df)} questions")
df.head(3)

In [8]:
# Run RAG inference: retrieve relevant law text, then ask GPT
system_prompt = """Du bist ein österreichischer Steuerrechtsexperte.
Nutze die folgenden Gesetzestexte als Kontext und beantworte die Frage präzise.
Nenne die relevanten Paragraphen (z.B. § 7 Abs. 1 KStG).
Antworte auf Deutsch in 1–3 kurzen Sätzen."""

answers = []

for i, row in df.iterrows():
    context_chunks = find_relevant_chunks(row["prompt"], k=3)
    context = "\n\n".join(context_chunks)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": f"Gesetzestext:\n{context}\n\nFrage: {row['prompt']}"}
        ],
        max_completion_tokens=300,
        temperature=0
    )
    answer = response.choices[0].message.content.strip()
    answers.append(answer)
    print(f"{i+1}/{len(df)} | {row['id']} | {answer[:70]}...")

1/643 | CORP-TAX-001 | Die steuerliche Bemessungsgrundlage für die Körperschaftsteuer ist das...
2/643 | CORP-TAX-002 | Wenn eine Körperschaft ein zinsloses Darlehen an ihren Gesellschafter ...
3/643 | CORP-TAX-003 | Gemäß § 15 Abs. 3 Z 3 EStG sind Körperschaften, die als Mitunternehmer...
4/643 | CORP-TAX-004 | (a) Bei der Tochtergesellschaft unterliegt die ausgeschüttete Dividend...
5/643 | CORP-TAX-005 | Die steuerliche Behandlung einer offenen Ausschüttung erfolgt gemäß § ...
6/643 | CORP-TAX-006 | Der maximal mögliche Verlustabzug in einem Veranlagungsjahr beträgt 75...
7/643 | CORP-TAX-007 | Ein Mantelkauf bezeichnet den Erwerb einer Gesellschaft, die keine akt...
8/643 | CORP-TAX-008 | Bei einem Forderungsverzicht eines Gesellschafters ist der nicht mehr ...
9/643 | CORP-TAX-009 | Die Verlustverrechnung von ausländischen Gruppenmitgliedern unterschei...
10/643 | CORP-TAX-010 | Steuerumlagen zwischen Gruppenträgern und Gruppenmitgliedern sind gemä...
11/643 | CORP-TAX-011 | Absch

In [9]:
# Save results
results = pd.DataFrame({"id": df["id"], "answer": answers})
results.to_csv("../results/model3_results.csv", index=False)
print(f"Saved {len(results)} answers to model3_results.csv")
results.head(3)

Saved 643 answers to model3_results.csv


,id,answer
0,CORP-TAX-001,Die steuerliche Bemessungsgrundlage für die Kö...
1,CORP-TAX-002,Wenn eine Körperschaft ein zinsloses Darlehen ...
2,CORP-TAX-003,Gemäß § 15 Abs. 3 Z 3 EStG sind Körperschaften...
